In [1]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/capstone-ml'
DATA = f'{ROOT}/data'
PRED = f'{ROOT}/predictions'
MODELS = f'{ROOT}/models'
LOGS = f'{ROOT}/logs'

import sys, importlib.metadata as md, torch
print('Python', sys.version.split()[0])
for p in ['torch', 'transformers', 'accelerate', 'datasets', 'scikit-learn', 'pandas', 'numpy']:
    print(f'{p}=={md.version(p)}')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE, switch the runtime to T4')

Mounted at /content/drive
Python 3.13.16
torch==2.14.1+cu130
transformers==5.22.1
accelerate==1.15.0
datasets==4.8.5
scikit-learn==1.6.1
pandas==2.2.3
numpy==2.4.6
GPU: Tesla T4


In [2]:
import json, os
import pandas as pd

splits = {s: pd.read_csv(f'{DATA}/sst5_{s}.csv') for s in ['train', 'val', 'test']}
assert splits['train']['label'].value_counts().sort_index().tolist() == [3310, 1624, 3610]

with open(f'{DATA}/class_weights.json') as f:
    CLASS_WEIGHTS = {int(k): v for k, v in json.load(f)['weights'].items()}
print({s: len(df) for s, df in splits.items()}, CLASS_WEIGHTS)

{'train': 8544, 'val': 1101, 'test': 2210} {0: 0.8604, 1: 1.7537, 2: 0.7889}


In [3]:
from transformers import AutoTokenizer

CKPT = 'distilbert-base-uncased'
tok = AutoTokenizer.from_pretrained(CKPT)

examples = ["it doesn't quite work, but the fundi was punctual and courteous.",
            "overcharged for the rewiring and left without explaining the invoice.",
            splits['train']['text'][2]]
for t in examples:
    print(t)
    print('  ->', tok.tokenize(t), '\n')

ids = tok(splits['train']['text'].tolist())['input_ids']
print(f'Longest training sentence: {max(len(i) for i in ids)} tokens')
print(f'Unknown tokens ([UNK]) in all of training: {sum(i.count(tok.unk_token_id) for i in ids)}')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

it doesn't quite work, but the fundi was punctual and courteous.
  -> ['it', 'doesn', "'", 't', 'quite', 'work', ',', 'but', 'the', 'fund', '##i', 'was', 'pun', '##ct', '##ual', 'and', 'court', '##eous', '.'] 

overcharged for the rewiring and left without explaining the invoice.
  -> ['over', '##charged', 'for', 'the', 're', '##wi', '##ring', 'and', 'left', 'without', 'explaining', 'the', 'in', '##vo', '##ice', '.'] 

they presume their audience won't sit still for a sociology lesson, however entertainingly presented, so they trot out the conventional science-fiction elements of bug-eyed monsters and futuristic women in skimpy clothes.
  -> ['they', 'pre', '##sume', 'their', 'audience', 'won', "'", 't', 'sit', 'still', 'for', 'a', 'sociology', 'lesson', ',', 'however', 'entertaining', '##ly', 'presented', ',', 'so', 'they', 'tr', '##ot', 'out', 'the', 'conventional', 'science', '-', 'fiction', 'elements', 'of', 'bug', '-', 'eyed', 'monsters', 'and', 'futuristic', 'women', 'in', 'ski',

In [4]:
from datasets import Dataset

MAX_LEN = 128

def to_hf(df):
    ds = Dataset.from_pandas(df[['id', 'text', 'label']])
    return ds.map(lambda batch: tok(batch['text'], truncation=True, max_length=MAX_LEN), batched=True)

hf = {s: to_hf(df) for s, df in splits.items()}
print(hf['train'])

Map:   0%|          | 0/8544 [00:00<?, ? examples/s]

Map:   0%|          | 0/1101 [00:00<?, ? examples/s]

Map:   0%|          | 0/2210 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 8544
})


In [5]:
import math, time, shutil
import numpy as np
import torch
import torch.nn.functional as F
from transformers import (AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding, set_seed)
from sklearn.metrics import f1_score, accuracy_score

ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}
WEIGHTS = torch.tensor([CLASS_WEIGHTS[i] for i in range(3)], dtype=torch.float)
BATCH, EPOCHS = 16, 4
STEPS_PER_EPOCH = math.ceil(len(hf['train']) / BATCH)
WARMUP = int(0.1 * STEPS_PER_EPOCH * EPOCHS)
print(f'{STEPS_PER_EPOCH} steps per epoch, {STEPS_PER_EPOCH * EPOCHS} in total, {WARMUP} warm-up steps')

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs['labels']
        outputs = model(**{k: v for k, v in inputs.items() if k != 'labels'})
        loss = F.cross_entropy(outputs.logits.float(), labels, weight=WEIGHTS.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'macro_f1': f1_score(labels, preds, average='macro'), 'accuracy': accuracy_score(labels, preds)}

collator = DataCollatorWithPadding(tok)

def make_trainer(seed, out_dir, **overrides):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        CKPT, num_labels=3, id2label=ID2LABEL, label2id={v: k for k, v in ID2LABEL.items()})
    settings = dict(output_dir=out_dir, learning_rate=2e-5, per_device_train_batch_size=BATCH,
                    per_device_eval_batch_size=64, num_train_epochs=EPOCHS, weight_decay=0.01,
                    warmup_steps=WARMUP, fp16=True, eval_strategy='epoch', save_strategy='epoch',
                    logging_strategy='epoch', load_best_model_at_end=True, metric_for_best_model='macro_f1',
                    greater_is_better=True, save_total_limit=2, seed=seed, report_to='none')
    settings.update(overrides)
    return WeightedTrainer(model=model, args=TrainingArguments(**settings), train_dataset=hf['train'],
                           eval_dataset=hf['val'], data_collator=collator, processing_class=tok,
                           compute_metrics=compute_metrics)

534 steps per epoch, 2136 in total, 213 warm-up steps


In [6]:
smoke = make_trainer(0, '/content/smoke', max_steps=40, warmup_steps=4, eval_strategy='no',
                     save_strategy='no', logging_strategy='steps', logging_steps=10, load_best_model_at_end=False)
start = time.perf_counter()
smoke.train()
sec_per_step = (time.perf_counter() - start) / 40
print(f'\n{sec_per_step:.2f} s per step -> about {sec_per_step * STEPS_PER_EPOCH * EPOCHS / 60:.0f} min of training per seed')
print('Validation after 40 steps:', {k: round(v, 4) for k, v in smoke.evaluate().items() if k in ('eval_macro_f1', 'eval_loss')})

free_gb = shutil.disk_usage(ROOT).free / 1e9
print(f'Free space reported for Drive: {free_gb:.1f} GB')
del smoke; torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
10,1.100348
20,1.090298
30,1.092392
40,1.072010



0.09 s per step -> about 3 min of training per seed


Training Loss,Validation Loss,Step,Macro F1,Accuracy
1.072010,1.082985,40,0.312581,0.470481


Validation after 40 steps: {'eval_loss': 1.083, 'eval_macro_f1': 0.3126}
Free space reported for Drive: 15.5 GB


In [7]:
from datetime import date

CKPT_DIR = f'{ROOT}/checkpoints'
os.makedirs(PRED, exist_ok=True)
log = f'{LOGS}/experiments.csv'

def log_run(row):
    logged = pd.read_csv(log)
    logged = logged[logged['run_id'].astype(str) != row['run_id']]
    pd.concat([logged, pd.DataFrame([row])]).to_csv(log, index=False)

def prediction_frame(split, proba, seed):
    df = splits[split]
    return pd.DataFrame({
        'id': df['id'], 'true_label': df['label'], 'pred_label': proba.argmax(axis=1),
        'p_neg': proba[:, 0].round(6), 'p_neu': proba[:, 1].round(6), 'p_pos': proba[:, 2].round(6),
        'model': 'distilbert', 'seed': seed, 'train_data': 'sst5', 'eval_set': f'sst5_{split}',
    })

results = []
for seed in [42, 43, 44]:
    out_dir = f'{CKPT_DIR}/distilbert_seed{seed}'
    final_dir = f'{MODELS}/distilbert_seed{seed}'
    test_file = f'{PRED}/distilbert_seed{seed}_sst5_test.csv'

    if os.path.exists(test_file) and not os.path.isdir(out_dir):
        print(f'Seed {seed} already finished, skipping')
        continue

    trainer = make_trainer(seed, out_dir)
    resume = os.path.isdir(out_dir) and any(d.startswith('checkpoint-') for d in os.listdir(out_dir))
    if resume:
        print(f'Seed {seed}: resuming from the last checkpoint on Drive')
    start = time.perf_counter()
    trainer.train(resume_from_checkpoint=True if resume else None)
    minutes = (time.perf_counter() - start) / 60

    evals = [h for h in trainer.state.log_history if 'eval_macro_f1' in h]
    for h in evals:
        print(f"  seed {seed} epoch {h['epoch']:.0f}: val loss {h['eval_loss']:.4f}, val macro-F1 {h['eval_macro_f1']:.4f}")
    best_epoch = int(max(evals, key=lambda h: h['eval_macro_f1'])['epoch'])

    trainer.save_model(final_dir)
    scores = {}
    for s in ['val', 'test']:
        logits = trainer.predict(hf[s]).predictions
        proba = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
        prediction_frame(s, proba, seed).to_csv(f'{PRED}/distilbert_seed{seed}_sst5_{s}.csv', index=False)
        scores[s] = f1_score(splits[s]['label'], proba.argmax(axis=1), average='macro')

    log_run({'run_id': f'distilbert-sst5-seed{seed}', 'model': 'distilbert', 'seed': seed, 'train_data': 'sst5',
             'hyperparameters': f'distilbert-base-uncased; adamw 2e-5; batch 16; wd 0.01; warmup 10%; fp16; '
                                f'weighted ce; max len {MAX_LEN}; 4 epochs, best by val macro-F1 (epoch {best_epoch})',
             'val_macro_f1': round(scores['val'], 4), 'test_macro_f1': round(scores['test'], 4),
             'date': date.today().isoformat()})
    shutil.rmtree(out_dir)
    results.append({'seed': seed, 'best_epoch': best_epoch, 'val_macro_f1': round(scores['val'], 4),
                    'test_macro_f1': round(scores['test'], 4), 'minutes': round(minutes, 1)})
    print(results[-1], '\n')
    del trainer; torch.cuda.empty_cache()

res = pd.DataFrame(results)
print(res.to_string(index=False))
print(f"\nTest macro-F1: mean {res['test_macro_f1'].mean():.4f}, std {res['test_macro_f1'].std():.4f}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.892258,0.800294,0.649212,0.711172
2,0.655094,0.813561,0.655507,0.712988
3,0.474191,0.941539,0.657549,0.702089
4,0.346940,1.024509,0.654565,0.703906


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (20 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  seed 44 epoch 1: val loss 0.7974, val macro-F1 0.6545
  seed 44 epoch 2: val loss 0.7994, val macro-F1 0.6636
  seed 44 epoch 3: val loss 0.9367, val macro-F1 0.6851
  seed 44 epoch 4: val loss 1.0437, val macro-F1 0.6832


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'seed': 44, 'best_epoch': 3, 'val_macro_f1': 0.6851, 'test_macro_f1': 0.6684, 'minutes': 2.6} 

 seed  best_epoch  val_macro_f1  test_macro_f1  minutes
   42           3        0.6575         0.6701      2.4
   43           2        0.6647         0.6541      2.5
   44           3        0.6851         0.6684      2.6

Test macro-F1: mean 0.6642, std 0.0088


In [8]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

runs = [('svm', '', f'{PRED}/svm_sst5_test.csv')] + \
       [('lstm', s, f'{PRED}/lstm_seed{s}_sst5_test.csv') for s in [42, 43, 44]] + \
       [('distilbert', s, f'{PRED}/distilbert_seed{s}_sst5_test.csv') for s in [42, 43, 44]]

rows = []
for model_name, seed, path in runs:
    p = pd.read_csv(path)
    prec, rec, f1, _ = precision_recall_fscore_support(p['true_label'], p['pred_label'], labels=[0, 1, 2], zero_division=0)
    rows.append({'model': model_name, 'seed': seed, 'neg_f1': f1[0], 'neu_recall': rec[1],
                 'neu_precision': prec[1], 'neu_f1': f1[1], 'pos_f1': f1[2], 'macro_f1': f1.mean()})
table = pd.DataFrame(rows)
print(table.round(4).to_string(index=False))

cols = ['neg_f1', 'neu_recall', 'neu_precision', 'neu_f1', 'pos_f1', 'macro_f1']
summary = table.groupby('model', sort=False)[cols].agg(['mean', 'std']).round(4)
print('\nMean and std across seeds (the SVM has one run, so no std):')
print(summary.to_string())

d = table[table['model'] == 'distilbert']
mid = int(d.iloc[(d['macro_f1'] - d['macro_f1'].mean()).abs().argmin()]['seed'])
p = pd.read_csv(f'{PRED}/distilbert_seed{mid}_sst5_test.csv')
print(f'\nDistilBERT seed {mid}, test confusion matrix (rows = true, columns = predicted):')
print(pd.DataFrame(confusion_matrix(p['true_label'], p['pred_label']),
                   index=['true neg', 'true neu', 'true pos'], columns=['pred neg', 'pred neu', 'pred pos']))

     model seed  neg_f1  neu_recall  neu_precision  neu_f1  pos_f1  macro_f1
       svm       0.6920      0.1414         0.2880  0.1897  0.7382    0.5399
      lstm   42  0.7324      0.1645         0.3832  0.2302  0.7380    0.5669
      lstm   43  0.6373      0.4473         0.2727  0.3389  0.7392    0.5718
      lstm   44  0.6341      0.4807         0.2812  0.3548  0.7355    0.5748
distilbert   42  0.7563      0.4653         0.3606  0.4063  0.8477    0.6701
distilbert   43  0.7550      0.4216         0.3287  0.3694  0.8379    0.6541
distilbert   44  0.7884      0.4036         0.3568  0.3788  0.8379    0.6684

Mean and std across seeds (the SVM has one run, so no std):
            neg_f1         neu_recall         neu_precision          neu_f1          pos_f1         macro_f1        
              mean     std       mean     std          mean     std    mean     std    mean     std     mean     std
model                                                                                    